# 09 — Rank Renewable Availability

## Research purpose

Rank hourly renewable availability within each UTC day to identify candidate hours for sending and receiving workload.


## Imports

Load the libraries for table processing and numerical validation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Project paths

Locate the project folder and define the paths for hourly renewable profiles, the query-location list and the ranked table.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
INPUT_PATH = DATA_INTERIM / "renewable_availability_cluster_hourly.parquet"
QUERY_LOCATIONS_PATH = DATA_INTERIM / "renewables_ninja_query_locations.csv"
OUTPUT_PATH = DATA_INTERIM / "renewable_availability_cluster_ranked_hourly.parquet"

## Ranking settings

Select the 2023–2025 weather years and set the candidate share to 25%, giving six hours per complete UTC day. Retain full daily ranks for the six-hour and nine-hour alternatives in notebook 12.


In [3]:
ACTIVE_WEATHER_YEARS = [2023, 2024, 2025]
CANDIDATE_SHARE = 0.25

## Define coverage checks

Define a validator for complete, unique UTC hourly coverage across every requested location and weather year. Check that solar, wind and index values are finite and that the index equals the 50/50 solar-wind mean.


In [4]:
def validate_renewable_coverage(frame, query_ids, weather_years):
    """Require complete paired hourly profiles for the currently requested locations."""
    if frame.empty or frame["query_location_id"].isna().any():
        raise ValueError("Renewable input must contain identified query-location rows")
    if set(frame["query_location_id"]) != set(query_ids):
        raise ValueError("Renewable input does not cover the current query locations")
    if frame.duplicated(["query_location_id", "time_utc"]).any():
        raise ValueError("Renewable input contains duplicate query-location UTC hours")
    factors = frame[["solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index"]]
    if not np.isfinite(factors.to_numpy(dtype=float)).all():
        raise ValueError("Each renewable hour requires finite PV, wind and index values")
    expected_index = 0.5 * frame["solar_capacity_factor"] + 0.5 * frame["wind_capacity_factor"]
    if not np.allclose(frame["renewable_availability_index"], expected_index, rtol=0, atol=1e-12):
        raise ValueError("Renewable index does not match the declared 50/50 PV-wind mean")

    expected_keys = {(query_id, year) for query_id in query_ids for year in weather_years}
    grouped = frame.groupby(["query_location_id", "weather_year"], sort=False)
    if set(grouped.groups) != expected_keys:
        raise ValueError("Renewable location/year coverage is incomplete")
    calendars = {
        year: pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", inclusive="left", tz="UTC")
        for year in weather_years
    }
    for (query_id, year), group in grouped:
        observed = pd.DatetimeIndex(group["time_utc"]).sort_values()
        if not observed.equals(calendars[year]):
            raise ValueError(f"Incomplete or off-hour UTC calendar for {query_id}, {year}")

## Load and validate hourly profiles

Read the hourly profiles, standardize time and numeric fields and select the configured weather years. Load the current query identifiers and apply the coverage checks before inspecting the available locations and sample records.


In [5]:
required_columns = {
    "query_location_id",
    "country_code",
    "country",
    "timezone",
    "renewable_query_region",
    "representative_lat",
    "representative_lon",
    "campus_count",
    "cluster_capacity_mw",
    "cluster_it_load_mw",
    "weather_year",
    "time_utc",
    "time_local",
    "solar_capacity_factor",
    "wind_capacity_factor",
    "renewable_availability_index",
}

renewable = pd.read_parquet(INPUT_PATH)
missing = sorted(required_columns - set(renewable.columns))
if missing:
    raise ValueError(f"Missing required renewable columns: {missing}")

renewable["weather_year"] = pd.to_numeric(renewable["weather_year"], errors="coerce").astype("Int64")
renewable["time_utc"] = pd.to_datetime(renewable["time_utc"], utc=True, errors="coerce")
if renewable["weather_year"].isna().any() or renewable["time_utc"].isna().any():
    raise ValueError("Renewable input contains an invalid weather year or UTC timestamp")
if not renewable["weather_year"].eq(renewable["time_utc"].dt.year).all():
    raise ValueError("Renewable weather years disagree with their UTC timestamps")
renewable = renewable[renewable["weather_year"].isin(ACTIVE_WEATHER_YEARS)].copy()
renewable["time_local"] = pd.to_datetime(renewable["time_local"], errors="coerce")
for column in ["solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index"]:
    renewable[column] = pd.to_numeric(renewable[column], errors="coerce")

query_locations = pd.read_csv(QUERY_LOCATIONS_PATH, usecols=["query_location_id"])
if query_locations["query_location_id"].isna().any() or query_locations["query_location_id"].duplicated().any():
    raise ValueError("Query-location identifiers must be present and unique")
validate_renewable_coverage(renewable, query_locations["query_location_id"], ACTIVE_WEATHER_YEARS)

print(f"Rows loaded: {len(renewable):,}")
print(f"Query locations: {renewable['query_location_id'].nunique():,}")
print(f"Weather years: {sorted(renewable['weather_year'].dropna().astype(int).unique().tolist())}")
print(renewable[["query_location_id", "weather_year", "time_utc", "renewable_availability_index"]].head(5).to_string(index=False))

Rows loaded: 2,288,448
Query locations: 87
Weather years: [2023, 2024, 2025]
query_location_id  weather_year                  time_utc  renewable_availability_index
  RN-ARE-DUBAI-01          2023 2023-01-01 00:00:00+00:00                        0.0460
  RN-ARE-DUBAI-01          2023 2023-01-01 01:00:00+00:00                        0.0660
  RN-ARE-DUBAI-01          2023 2023-01-01 02:00:00+00:00                        0.0835
  RN-ARE-DUBAI-01          2023 2023-01-01 03:00:00+00:00                        0.1200
  RN-ARE-DUBAI-01          2023 2023-01-01 04:00:00+00:00                        0.2100


## Rank daily hours and select candidates

Rank each location's hours from highest to lowest and lowest to highest renewable availability within each UTC day, resolving ties by ascending timestamp. Calculate rank percentiles and flag the six highest hours for receiving workload and the six lowest for sending workload, while retaining full daily ranks.


In [6]:
renewable = renewable.sort_values(["query_location_id", "weather_year", "time_utc"]).reset_index(drop=True)
renewable["utc_date"] = renewable["time_utc"].dt.date.astype(str)
renewable["utc_hour"] = renewable["time_utc"].dt.hour.astype("Int64")
renewable["local_date"] = renewable["time_local"].dt.date.astype(str)
renewable["local_hour"] = renewable["time_local"].dt.hour.astype("Int64")
renewable["day_of_week_utc"] = renewable["time_utc"].dt.dayofweek.astype("Int64")
renewable["is_weekend_utc"] = renewable["day_of_week_utc"].isin([5, 6])

group_cols = ["query_location_id", "weather_year", "utc_date"]
renewable["daily_hour_count"] = renewable.groupby(group_cols)["time_utc"].transform("count").astype("Int64")

receive_order = renewable.sort_values(group_cols + ["renewable_availability_index", "time_utc"], ascending=[True, True, True, False, True])
renewable.loc[receive_order.index, "receive_rank_daily"] = receive_order.groupby(group_cols).cumcount() + 1

send_order = renewable.sort_values(group_cols + ["renewable_availability_index", "time_utc"], ascending=[True, True, True, True, True])
renewable.loc[send_order.index, "send_rank_daily"] = send_order.groupby(group_cols).cumcount() + 1

renewable["receive_rank_daily"] = renewable["receive_rank_daily"].astype("Int64")
renewable["send_rank_daily"] = renewable["send_rank_daily"].astype("Int64")
rank_denominator = (renewable["daily_hour_count"].astype(float) - 1)
renewable["renewable_rank_percentile"] = 1 - ((renewable["receive_rank_daily"].astype(float) - 1) / rank_denominator)

candidate_hours = np.ceil(renewable["daily_hour_count"].astype(float) * CANDIDATE_SHARE).astype(int)
renewable["receive_candidate"] = renewable["receive_rank_daily"].astype(int) <= candidate_hours
renewable["send_candidate"] = renewable["send_rank_daily"].astype(int) <= candidate_hours

print(renewable[["send_candidate", "receive_candidate"]].sum().to_string())
print(renewable[["query_location_id", "weather_year", "utc_date", "time_utc", "renewable_availability_index", "receive_rank_daily", "send_rank_daily", "receive_candidate", "send_candidate"]].head(8).to_string(index=False))

send_candidate       572112
receive_candidate    572112
query_location_id  weather_year   utc_date                  time_utc  renewable_availability_index  receive_rank_daily  send_rank_daily  receive_candidate  send_candidate
  RN-ARE-DUBAI-01          2023 2023-01-01 2023-01-01 00:00:00+00:00                        0.0460                  18                7              False           False
  RN-ARE-DUBAI-01          2023 2023-01-01 2023-01-01 01:00:00+00:00                        0.0660                  16                9              False           False
  RN-ARE-DUBAI-01          2023 2023-01-01 2023-01-01 02:00:00+00:00                        0.0835                  15               10              False           False
  RN-ARE-DUBAI-01          2023 2023-01-01 2023-01-01 03:00:00+00:00                        0.1200                  12               13              False           False
  RN-ARE-DUBAI-01          2023 2023-01-01 2023-01-01 04:00:00+00:00                     

## Validate and save ranked profiles

Check that every UTC day has 24 hours, complete rank ranges and the expected number of send and receive candidates. Save the ranked profiles as Parquet and display a sample of the resulting ranks and flags.


In [7]:
daily = renewable.groupby(group_cols, dropna=False).agg(
    rows=("time_utc", "count"),
    receive_min=("receive_rank_daily", "min"),
    receive_max=("receive_rank_daily", "max"),
    send_min=("send_rank_daily", "min"),
    send_max=("send_rank_daily", "max"),
    receive_candidates=("receive_candidate", "sum"),
    send_candidates=("send_candidate", "sum"),
).reset_index()

unexpected_daily_counts = daily[daily["rows"] != 24]
if len(unexpected_daily_counts):
    raise ValueError(f"{len(unexpected_daily_counts):,} query-location days do not contain 24 UTC hours")

if not (daily["receive_min"].eq(1).all() and daily["send_min"].eq(1).all()):
    raise ValueError("Daily ranks do not start at 1 for every group")
if not (daily["receive_max"].eq(daily["rows"]).all() and daily["send_max"].eq(daily["rows"]).all()):
    raise ValueError("Daily ranks do not cover all rows in every group")

expected_candidate_hours = np.ceil(daily["rows"] * CANDIDATE_SHARE).astype(int)
if not daily["receive_candidates"].eq(expected_candidate_hours).all() or not daily["send_candidates"].eq(expected_candidate_hours).all():
    raise ValueError("Daily candidate counts do not match the configured share")

output_columns = [
    "query_location_id",
    "country_code",
    "country",
    "timezone",
    "renewable_query_region",
    "representative_lat",
    "representative_lon",
    "campus_count",
    "cluster_capacity_mw",
    "cluster_it_load_mw",
    "weather_year",
    "time_utc",
    "time_local",
    "utc_date",
    "utc_hour",
    "local_date",
    "local_hour",
    "day_of_week_utc",
    "is_weekend_utc",
    "solar_capacity_factor",
    "wind_capacity_factor",
    "renewable_availability_index",
    "daily_hour_count",
    "receive_rank_daily",
    "send_rank_daily",
    "renewable_rank_percentile",
    "receive_candidate",
    "send_candidate",
    "pv_source_file",
    "wind_source_file",
]
output_columns = [column for column in output_columns if column in renewable.columns]
ranked = renewable[output_columns].sort_values(["query_location_id", "time_utc"]).reset_index(drop=True)
# Normalize numeric representation before storage for deterministic downstream use.
ranked["renewable_rank_percentile"] = pd.to_numeric(
    ranked["renewable_rank_percentile"].astype(str), errors="raise"
)
ranked.to_parquet(OUTPUT_PATH, index=False, compression="zstd")

ranked.head()

,query_location_id,country_code,country,timezone,renewable_query_region,representative_lat,representative_lon,campus_count,cluster_capacity_mw,cluster_it_load_mw,...,wind_capacity_factor,renewable_availability_index,daily_hour_count,receive_rank_daily,send_rank_daily,renewable_rank_percentile,receive_candidate,send_candidate,pv_source_file,wind_source_file
0,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,...,0.092,0.0460,24,18,7,0.260870,False,False,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
1,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,...,0.132,0.0660,24,16,9,0.347826,False,False,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
2,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,...,0.167,0.0835,24,15,10,0.391304,False,False,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
3,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,...,0.169,0.1200,24,12,13,0.521739,False,False,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
4,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,...,0.128,0.2100,24,9,16,0.652174,False,False,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
